# Chapter 8, Part 3: Evaluation & Guardrails

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-08/03_evaluation.ipynb)

**What you'll build:**
- Grounding validation
- Single-query evaluation with precision, recall, and latency
- Multi-turn conversation evaluation
- LLM-as-judge for explanation quality

**Prerequisites:** Run Parts 1 and 2 first.

In [ ]:
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu
setup_colab_environment()
check_gpu()

In [ ]:
import os, json, numpy as np, pandas as pd, faiss
from pathlib import Path

from recsys.data.loaders import load_movielens, load_movielens_descriptions
from recsys.fourstage_recsys.retrieval.itemknn_retrieval import ItemKNNRetrieval
from recsys.agentic.llm_client import LLMClient
from recsys.agentic.hybrid_retriever import HybridRetriever
from recsys.agentic.agent import MovieRecommenderAgent, Tool
from recsys.agentic.memory import ConversationalRecommender, UserProfiler
from recsys.agentic.guardrails import (
  GroundingValidator, LoopDetector,
  AgentEvaluator, llm_as_judge
)
DATA_PATH = get_data_path()

In [ ]:
# LLM Setup - same as Notebook 2
llm = LLMClient(
  backend="api",
  api_key=os.environ.get("GEMINI_API_KEY"),
  base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
  model="gemini-2.0-flash"
)
print(llm.generate(system_prompt="Reply in 5 words.", user_message="Hello"))

In [ ]:
# Rebuild the full system
ratings, movies = load_movielens(dataset='ml-25m', data_dir=DATA_PATH)
descriptions = load_movielens_descriptions(data_dir=DATA_PATH)
if 'overview' not in movies.columns and not descriptions.empty:
  movies = movies.merge(
    descriptions[['title', 'overview']].drop_duplicates('title'),
    on='title', how='left'
  )
  movies['overview'] = movies['overview'].fillna('')

embeddings = np.load(Path(DATA_PATH) / "movie_embeddings.npy")
index = faiss.IndexFlatIP(embeddings.shape[1])
emb_norm = embeddings.copy()
faiss.normalize_L2(emb_norm)
index.add(emb_norm.astype(np.float32))

ratings_sample = ratings.sample(n=min(500_000, len(ratings)), random_state=42)
item_knn = ItemKNNRetrieval(ratings_sample)
movies_reset = movies.reset_index(drop=True)

retriever = HybridRetriever(
  faiss_index=index, embeddings=emb_norm,
  movies_df=movies_reset, item_knn=item_knn
)

def search_movies(query, k=5):
  results = retriever.search(query, k=int(k))
  return [{"title": r["title"], "genres": r["genres"],
           "year": r.get("year")} for r in results]

def filter_movies(movies, genre=None, year_min=None, year_max=None):
  filtered = movies
  if genre:
    filtered = [m for m in filtered if genre.lower() in m.get("genres", "").lower()]
  if year_min:
    filtered = [m for m in filtered if m.get("year") and m["year"] >= int(year_min)]
  if year_max:
    filtered = [m for m in filtered if m.get("year") and m["year"] <= int(year_max)]
  return filtered

tools = [
  Tool("search_movies", "Search movies by description", search_movies,
       {"query": "str", "k": "int (default 5)"}),
  Tool("filter_movies", "Filter movies by metadata", filter_movies,
       {"movies": "list", "genre": "str", "year_min": "int", "year_max": "int"})
]

agent = MovieRecommenderAgent(llm=llm, tools=tools, max_steps=5)
conv_rec = ConversationalRecommender(llm=llm, retriever=retriever)
print("Full system ready.")

## 1. Grounding Validation

Are the recommended movies real?

In [ ]:
validator = GroundingValidator(movies_reset)

response = agent.run("Recommend mind-bending sci-fi movies", debug=True)
print("\nAgent response:")
print(response)

result = validator.validate(response)
print(f"\nGrounding:")
print(f"  Grounded: {result['grounded']}")
print(f"  Hallucinated: {result['hallucinated']}")
print(f"  Rate: {result['grounding_rate']:.0%}")

## 2. Single-Query Evaluation

In [ ]:
test_cases = [
  {"query": "Space exploration movies",
   "relevant_items": ["Interstellar", "The Martian", "Gravity",
                       "2001: A Space Odyssey", "Apollo 13"]},
  {"query": "Classic film noir from the 1940s",
   "relevant_items": ["Double Indemnity", "The Maltese Falcon",
                       "The Big Sleep", "Laura", "The Third Man"]},
  {"query": "Feel-good comedies from the 2010s",
   "relevant_items": ["The Grand Budapest Hotel", "Hunt for the Wilderpeople",
                       "What We Do in the Shadows", "Paddington 2"]}
]
print(f"Defined {len(test_cases)} test cases")

In [ ]:
catalog_titles = set(movies_reset['title'].str.lower())
evaluator = AgentEvaluator(catalog_titles=catalog_titles)

results = evaluator.evaluate_batch(agent, test_cases)

for r in results["individual"]:
  print(f"\nQuery: {r['query']}")
  print(f"  Precision: {r['precision']:.2f}  Recall: {r['recall']:.2f}")
  print(f"  Grounding: {r['grounding_rate']:.0%}  Latency: {r['latency_seconds']:.1f}s")
  if r['hallucinated']:
    print(f"  Hallucinated: {r['hallucinated']}")

agg = results["aggregate"]
print(f"\n=== Aggregate ===")
print(f"  Precision: {agg['avg_precision']:.2f}  Recall: {agg['avg_recall']:.2f}")
print(f"  Grounding: {agg['avg_grounding_rate']:.0%}  Latency: {agg['avg_latency_seconds']:.1f}s")

## 3. Multi-Turn Evaluation

In [ ]:
scenario = {
  "user_id": "test_user_1",
  "turns": [
    {"user_message": "Recommend a thriller",
     "assertions": [
       {"name": "gives_response", "fn": lambda r: len(r) > 50}
     ]},
    {"user_message": "Too violent. Something psychological instead.",
     "assertions": [
       {"name": "acknowledges_feedback",
        "fn": lambda r: any(w in r.lower() for w in
          ["psychological", "mind", "subtle", "got it", "instead"])}
     ]},
    {"user_message": "Something from the last 5 years",
     "assertions": [
       {"name": "respects_recency",
        "fn": lambda r: any(str(y) in r for y in range(2020, 2027))}
     ]}
  ]
}

conv_rec.reset()
turn_results = evaluator.evaluate_multiturn(conv_rec, scenario)

for tr in turn_results:
  print(f"\n--- Turn {tr['turn']} ---")
  print(f"User: {tr['user_message']}")
  print(f"Response: {tr['response'][:200]}...")
  for check in tr['checks']:
    status = 'PASS' if check['passed'] else 'FAIL'
    print(f"  {status}: {check['name']}")

## 4. LLM-as-Judge

In [ ]:
conv_rec.reset()
response = conv_rec.chat(
  "test_user_2",
  "Recommend a movie like Inception for someone who loves puzzles"
)
print("Agent response:")
print(response)

try:
  scores = llm_as_judge(
    llm=llm,
    query="movie like Inception for puzzle lovers",
    recommendation=response[:500],
    explanation=response[:500],
    user_profile="Enjoys complex narratives and mind-bending films"
  )
  print(f"\nLLM-as-Judge:")
  print(f"  Specificity:  {scores.get('specificity', 'N/A')}/5")
  print(f"  Accuracy:     {scores.get('accuracy', 'N/A')}/5")
  print(f"  Helpfulness:  {scores.get('helpfulness', 'N/A')}/5")
  print(f"  Reasoning:    {scores.get('reasoning', 'N/A')}")
except Exception as e:
  print(f"Judge failed: {e}")

## 5. Cost Analysis

In [ ]:
# Based on typical gpt-4o-mini pricing
INPUT_COST_PER_1K = 0.00015
OUTPUT_COST_PER_1K = 0.0006
avg_input = 1500
avg_output = 300
avg_steps = results["aggregate"]["avg_steps"] if results else 3

cost_per_query = avg_steps * (
  (avg_input / 1000) * INPUT_COST_PER_1K +
  (avg_output / 1000) * OUTPUT_COST_PER_1K
)

queries_per_day = 100_000
monthly = cost_per_query * queries_per_day * 30

print(f"Avg steps/query: {avg_steps:.1f}")
print(f"Cost/query: ${cost_per_query:.4f}")
print(f"At {queries_per_day:,} queries/day: ${monthly:,.0f}/month")
print(f"Traditional system: ~$500-1K/month")
print(f"Ratio: ~{monthly / 750:.0f}x more expensive")

## Summary

1. **Grounding validation** - every recommendation checked against catalog
2. **Single-query evaluation** - precision, recall, grounding, latency
3. **Multi-turn evaluation** - scripted scenarios with per-turn assertions
4. **LLM-as-judge** - automated scoring of explanation quality
5. **Cost analysis** - the latency/cost trade-off that determines deployment surface